# Solar Filament Segmentation - fast baseline (U-Net + PQ-tuned post-processing)
Run all cells top to bottom. Output: `submission.csv` in the same folder as this notebook.
Auto-detects GPU/CPU and adapts resolution, model size and training time.
**Only edit `ZIP_PATH` and (optionally) `TIME_BUDGET_MIN` in cell 2.**

# 1) install helper packages (PyTorch must already be installed - see instructions in chat)
%pip install -q segmentation-models-pytorch pycocotools opencv-python-headless pandas numpy tqdm

In [1]:
# 2) CONFIG
import os, json, glob, time, zipfile, itertools, csv, math
from collections import defaultdict
import numpy as np, cv2, pandas as pd, torch, torch.nn.functional as F
from tqdm.auto import tqdm
from pycocotools import mask as mu
import segmentation_models_pytorch as smp

ZIP_PATH = r"C:\Users\shrut\Desktop\final\filament-segmentation-2026.zip"
WORK_DIR = "./work"
TIME_BUDGET_MIN = None      # training minutes; None = auto (GPU 12 min, CPU 15 min)
SEED = 0

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    RES, ENCODER, BS = 512, "resnet34", 8
    BUDGET = (TIME_BUDGET_MIN or 12) * 60
else:
    RES, ENCODER, BS = 256, "resnet18", 8
    BUDGET = (TIME_BUDGET_MIN or 15) * 60
use_amp = device == "cuda"
print("device:", device, "| RES", RES, "| encoder", ENCODER, "| train budget (min):", BUDGET/60)
if device == "cuda": print(torch.cuda.get_device_name(0))
rng = np.random.default_rng(SEED); torch.manual_seed(SEED)

c:\Users\shrut\Desktop\final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu | RES 256 | encoder resnet18 | train budget (min): 15.0


In [2]:
# 3) unzip + locate files
data_dir = os.path.join(WORK_DIR, "data")
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(data_dir)
ann_path = glob.glob(os.path.join(data_dir, "**", "*Annotations*train*.json"), recursive=True)[0]
train_dir = [d for d in glob.glob(os.path.join(data_dir, "**", "train_images"), recursive=True) if os.path.isdir(d)][0]
test_dir  = [d for d in glob.glob(os.path.join(data_dir, "**", "test_images"),  recursive=True) if os.path.isdir(d)][0]
print(ann_path, train_dir, test_dir, sep="\n")
get_stem = lambda p: os.path.splitext(os.path.basename(p))[0][-16:]
train_files = {get_stem(p): p for p in glob.glob(os.path.join(train_dir, "*"))}
test_files  = {get_stem(p): p for p in sorted(glob.glob(os.path.join(test_dir, "*")))}
print(len(train_files), "train files,", len(test_files), "test files")

./work\data\MAGFiLO_1.0_Kaggle_2026\train\MAGFiLO_1.0_Annotations_kaggle2026_train.json
./work\data\MAGFiLO_1.0_Kaggle_2026\train\train_images
./work\data\MAGFiLO_1.0_Kaggle_2026\test\test_images
707 train files, 180 test files


In [3]:
# 4) build dataset: group duplicate annotator sets per image, soft-label = mean of annotators
coco = json.load(open(ann_path))
anns_by_img = defaultdict(list)
for a in coco["annotations"]: anns_by_img[a["image_id"]].append(a)
groups = defaultdict(list)
for im in coco["images"]:
    s = get_stem(im["file_name"])
    if s in train_files: groups[s].append(im)
stems = sorted(groups)
print("unique train images:", len(stems))

perm = rng.permutation(len(stems)); n_val = max(8, int(0.15*len(stems)))
val_stems = [stems[i] for i in perm[:n_val]]; tr_stems = [stems[i] for i in perm[n_val:]]

def poly_masks(img_info):
    w = img_info["width"]; sc = RES / w; inst = []
    for a in anns_by_img[img_info["id"]]:
        m = np.zeros((RES, RES), np.uint8)
        for seg in a["segmentation"]:
            pts = np.round(np.array(seg, dtype=np.float64).reshape(-1, 2) * sc).astype(np.int32)
            cv2.fillPoly(m, [pts], 1)
        inst.append(m)
    return inst

def load_img(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return cv2.resize(g, (RES, RES), interpolation=cv2.INTER_AREA)

def build(stem_list, keep_gt=False):
    X, Y, GT = [], [], []
    for s in tqdm(stem_list):
        X.append(load_img(train_files[s]))
        per_annot = [poly_masks(im) for im in sorted(groups[s], key=lambda d: str(d["id"]))]
        union = [np.clip(np.sum(p, 0), 0, 1) if len(p) else np.zeros((RES, RES), np.uint8) for p in per_annot]
        Y.append(np.mean(union, 0).astype(np.float32))
        if keep_gt: GT.append([m.astype(bool) for m in per_annot[0]])   # instances of first annotator
    return np.stack(X), np.stack(Y), GT

Xtr, Ytr, _ = build(tr_stems)
Xva, Yva, GTva = build(val_stems, keep_gt=True)
print(Xtr.shape, Xva.shape)

unique train images: 707


100%|██████████| 106/106 [00:02<00:00, 40.25it/s]

(601, 256, 256) (106, 256, 256)


In [4]:
# 5) model + training (time-based cosine LR, stops when budget is used)
try:
    model = smp.Unet(ENCODER, encoder_weights="imagenet", in_channels=1, classes=1)
except Exception as e:
    print("Could not download pretrained weights (", type(e).__name__, ") -> training from scratch")
    model = smp.Unet(ENCODER, encoder_weights=None, in_channels=1, classes=1)
model.to(device)
norm = lambda x: (x - 0.45) / 0.25

def get_batch(idx):
    xs, ys = Xtr[idx].copy(), Ytr[idx].copy()
    for j in range(len(idx)):
        if rng.random() < .5: xs[j] = xs[j][:, ::-1]; ys[j] = ys[j][:, ::-1]
        if rng.random() < .5: xs[j] = xs[j][::-1];    ys[j] = ys[j][::-1]
        k = int(rng.integers(4)); xs[j] = np.rot90(xs[j], k); ys[j] = np.rot90(ys[j], k)
    x = torch.from_numpy(xs).float().div(255).unsqueeze(1)
    c = torch.empty(len(idx), 1, 1, 1).uniform_(0.85, 1.15); b = torch.empty(len(idx), 1, 1, 1).uniform_(-0.05, 0.05)
    x = ((x - 0.5) * c + 0.5 + b).clamp(0, 1)
    return norm(x).to(device), torch.from_numpy(ys).unsqueeze(1).to(device)

def loss_fn(lg, y):
    bce = F.binary_cross_entropy_with_logits(lg, y)
    p = torch.sigmoid(lg); inter = (p*y).sum((1,2,3))
    dice = 1 - (2*inter + 1) / (p.sum((1,2,3)) + y.sum((1,2,3)) + 1)
    return bce + dice.mean()

BASE_LR = 5e-4
opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
t0 = time.time(); ep = 0; done = False
while not done:
    model.train(); order = rng.permutation(len(Xtr)); losses = []
    for i in range(0, len(order) - BS + 1, BS):
        frac = (time.time() - t0) / BUDGET
        if frac >= 1: done = True; break
        for g in opt.param_groups: g["lr"] = BASE_LR * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))
        x, y = get_batch(order[i:i+BS])
        with torch.autocast(device_type=device, enabled=use_amp):
            lg = model(x)
        loss = loss_fn(lg.float(), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        losses.append(loss.item())
    ep += 1
    print(f"epoch {ep} | loss {np.mean(losses) if losses else float('nan'):.4f} | {(time.time()-t0)/60:.1f} min")
torch.save(model.state_dict(), os.path.join(WORK_DIR, "model.pt"))

c:\Users\shrut\Desktop\final\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shrut\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\shrut\Desktop\final\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses sym

epoch 1 | loss 1.2950 | 1.5 min
epoch 2 | loss 0.9690 | 3.0 min
epoch 3 | loss 0.7092 | 4.4 min
epoch 4 | loss 0.5430 | 5.9 min
epoch 5 | loss 0.4924 | 7.4 min
epoch 6 | loss 0.4766 | 8.8 min
epoch 7 | loss 0.4572 | 10.3 min
epoch 8 | loss 0.4521 | 11.8 min
epoch 9 | loss 0.4417 | 13.2 min
epoch 10 | loss 0.4412 | 14.7 min
epoch 11 | loss 0.4497 | 15.0 min


In [5]:
# 6) inference helper + validation probabilities
@torch.no_grad()
def predict_probs(X):
    model.eval(); out = []
    for i in range(0, len(X), BS):
        x = norm(torch.from_numpy(X[i:i+BS]).float().div(255).unsqueeze(1)).to(device)
        with torch.autocast(device_type=device, enabled=use_amp):
            p = torch.sigmoid(model(x).float()) + torch.sigmoid(model(x.flip(-1)).float()).flip(-1)
        out.append((p / 2).squeeze(1).cpu().numpy())
    return np.concatenate(out)

Pva = predict_probs(Xva)
inter = ((Pva > .5) & (Yva > .5)).sum(); print("val pixel Dice @0.5:", 2*inter / ((Pva > .5).sum() + (Yva > .5).sum() + 1e-9))

val pixel Dice @0.5: 0.6422970112917551


In [6]:
# 7) post-processing (instances) + PQ metric + parameter search
def instances(prob, thr, k, min_area, s=1.0):
    m = (prob > thr).astype(np.uint8)
    kk = int(round(k * s))
    if kk > 1: m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kk, kk)))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    return [lab == i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area * s * s]

def pq_counts(gts, preds):
    if not gts or not preds: return 0.0, 0, len(preds), len(gts)
    G = np.stack([g.ravel() for g in gts]).astype(np.float32); P = np.stack([p.ravel() for p in preds]).astype(np.float32)
    inter = G @ P.T; iou = inter / (G.sum(1)[:, None] + P.sum(1)[None] - inter + 1e-9)
    match = iou > 0.5; tp = int(match.sum())
    return float(iou[match].sum()), tp, len(preds) - tp, len(gts) - tp

def eval_pq(thr, k, min_area):
    s_iou = tp = fp = fn = 0
    for prob, gt in zip(Pva, GTva):
        a, b, c, d = pq_counts(gt, instances(prob, thr, k, min_area))
        s_iou += a; tp += b; fp += c; fn += d
    return s_iou / max(tp + .5*fp + .5*fn, 1e-9)

best = (-1, None)
grid = list(itertools.product([0.3, 0.4, 0.5, 0.6], [0, 3, 5, 9], [20, 60, 150]))
for thr, k, ma in tqdm(grid):
    pq = eval_pq(thr, k, ma)
    if pq > best[0]: best = (pq, (thr, k, ma))
print("best val PQ: %.4f with (thr, close_k, min_area) = %s" % best)
THR, K, MINA = best[1]

100%|██████████| 48/48 [00:07<00:00,  6.30it/s]

best val PQ: 0.2755 with (thr, close_k, min_area) = (0.5, 0, 20)


In [7]:
# 8) predict test set and write submission.csv
rows = []; n_img_with = 0
for stem, path in tqdm(test_files.items()):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE); H, W = g.shape
    x = cv2.resize(g, (RES, RES), interpolation=cv2.INTER_AREA)[None]
    prob = cv2.resize(predict_probs(x)[0], (W, H), interpolation=cv2.INTER_LINEAR)
    insts = instances(prob, THR, K, MINA, s=W / RES)
    if not insts:   # fallback: keep the single biggest blob at a lower threshold
        insts = sorted(instances(prob, 0.25, 0, MINA / 2, s=W / RES), key=lambda m: -m.sum())[:1]
    n_img_with += bool(insts)
    for i, m in enumerate(insts, 1):
        r = mu.encode(np.asfortranarray(m.astype(np.uint8)))
        rows.append((f"{stem}_{i}", r["counts"].decode()))
sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
sub.to_csv("submission.csv", index=False, quoting=csv.QUOTE_NONE)
print(len(sub), "filaments over", n_img_with, "/", len(test_files), "images -> submission.csv")
sub.head()

100%|██████████| 180/180 [00:38<00:00,  4.63it/s]


957 filaments over 178 / 180 images -> submission.csv


,filament_id,segmentation_rle
0,20110120105534Ch_1,g_bh29do16L3M200O1O2O2M1O2O0O10000O3N1O1N2O0O0...
1,20110120105534Ch_2,ciXQ24jo17H4I7I6N3M2N2O1O1N2O1M2O2O100O00000O1...
2,20110130110334Ch_1,[Xlm12go1?F7I5L2N2N2O1M3O2M4L3N2M2O100O1O2O1N2...
3,20110130110334Ch_2,i_fd24eo19M5J5L4L5L3L2O2N10001N3N2M2O1N10O11O1...
4,20110214175414Mh_1,T]^[26bo1>H5L3M2N101M2N3ZOe0M3N3O0O1O1O2M2O2M2...
